# Đánh giá Hệ thống Acne Advisor AI bằng RAGAS

Notebook này chạy đánh giá định lượng toàn diện hệ thống **Acne Advisor AI** theo phương pháp **RAGAS** (*Retrieval Augmented Generation Assessment*, Es et al., 2023, `arXiv:2309.15217`).

### Các chỉ số được đo:
1. **Faithfulness** (*Độ trung thực với bằng chứng*): phát hiện ảo giác (hallucination).
2. **Answer Relevancy** (*Độ liên quan câu trả lời*): đo mức độ bám sát câu hỏi người dùng.
3. **Context Precision** (*Độ chính xác ngữ cảnh*): đo độ chính xác xếp hạng top ngữ cảnh truy hồi.
4. **Context Recall** (*Độ bao phủ ngữ cảnh*): đo ngữ cảnh truy hồi có đủ ý theo đáp án chuẩn.
5. **Factual Correctness** (*Độ chính xác nội dung*): so khớp sự kiện với đáp án chuẩn.
6. **Safe Abstention Accuracy** (*Tỷ lệ từ chối an toàn*): kiểm tra khả năng từ chối đúng ca ngoài phạm vi / thiếu bằng chứng.

Kết quả chi tiết được hiển thị trực tiếp ngay bên dưới các cell và đồng thời xuất ra **`evaluation/results/ragas_evaluation_report.md`**.

In [1]:
# 1. Khởi tạo đường dẫn dự án và nạp biến môi trường (.env)
from __future__ import annotations
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'evaluation':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

load_dotenv(PROJECT_ROOT / '.env')
print('Project root:', PROJECT_ROOT)
print('GOOGLE_API_KEY configured:', bool(os.getenv('GOOGLE_API_KEY')))
print('QDRANT_URL:', os.getenv('QDRANT_URL', 'http://127.0.0.1:6333'))

Project root: C:\Study\Acne Advisor AI\acne-agent-system
GOOGLE_API_KEY configured: True
QDRANT_URL: http://localhost:6333


In [2]:
# 2. Cấu hình tham số đánh giá và kiểm tra Knowledge Build hiện hành
from IPython.display import Markdown, display
import pandas as pd
from evaluation.ragas_runner import (
    DEFAULT_DATASET_PATH,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_RESULTS_DIR,
    export_evaluation_artifacts,
    load_evaluation_dataset,
    render_markdown_report,
    run_system_on_dataset,
    score_records_with_ragas,
    summarize_evaluation,
)
from src.knowledge.versioning import resolve_active_knowledge_build_id

DATASET_PATH = PROJECT_ROOT / DEFAULT_DATASET_PATH
RESULTS_DIR = PROJECT_ROOT / DEFAULT_RESULTS_DIR
JUDGE_PROVIDER = 'google'
JUDGE_MODEL = 'gemini-3.1-flash-lite'  # Model độc lập với generator (gemini-3.5-flash-lite)
EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL
REQUEST_DELAY_SECONDS = 2.5  # Giãn nhịp để tránh vượt giới hạn 15 RPM của Gemini Free Tier
MAX_CASES = None  # Đặt số nguyên (vd: 4) để chạy thử nhanh; None để chạy toàn bộ bộ dữ liệu.

kb_build_id = resolve_active_knowledge_build_id()
print('Active KB Build ID:', kb_build_id)
print('Judge Provider:', JUDGE_PROVIDER, '| Judge LLM:', JUDGE_MODEL, '| Embedding:', EMBEDDING_MODEL)

Active KB Build ID: b768fae999391f47760a
Judge Provider: google | Judge LLM: gemini-3.1-flash-lite | Embedding: models/gemini-embedding-2


In [3]:
# 3. Nạp và xem tổng quan bộ câu hỏi đánh giá
dataset_meta, cases = load_evaluation_dataset(DATASET_PATH)
if MAX_CASES is not None:
    cases = cases[:MAX_CASES]

df_cases = pd.DataFrame([
    {'case_id': c.case_id, 'category': c.category, 'expected': c.expected_behavior, 'question': c.user_input}
    for c in cases
])
print(f'Tổng số câu hỏi sẽ chạy: {len(cases)}')
display(df_cases)

Tổng số câu hỏi sẽ chạy: 20


,case_id,category,expected,question
0,SYM-01,symptoms_and_types,answer,Mụn đầu trắng là gì và biểu hiện như thế nào t...
1,SYM-02,symptoms_and_types,answer,Mụn đầu đen hình thành như thế nào và nhận biế...
2,SYM-03,symptoms_and_types,answer,Các dạng tổn thương viêm của mụn trứng cá bao ...
3,CAU-01,causes_and_mechanisms,answer,Nguyên nhân chính gây ra mụn trứng cá là gì?
4,CAU-02,causes_and_mechanisms,answer,Thế nào là da dầu và vì sao da dầu dễ bị mụn t...
5,CAU-03,causes_and_mechanisms,answer,Mụn trứng cá thường gặp nhất ở độ tuổi nào và ...
6,CMP-01,comparison,answer,Mụn đầu trắng khác gì với mụn đầu đen? Hãy so ...
7,CMP-02,comparison,answer,Sự khác nhau giữa mụn không viêm và mụn viêm l...
8,TRT-01,treatment_and_actives,answer,Benzoyl peroxide có tác dụng gì trong điều trị...
9,TRT-02,treatment_and_actives,answer,Adapalene thuộc nhóm hoạt chất nào và có vai t...


In [4]:
# 4. Chạy hệ thống Acne Advisor AI trên bộ câu hỏi (tắt cache để đo thực tế)
system_records = await run_system_on_dataset(cases)
df_sys = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'action': r['action'],
        'contexts': len(r['retrieved_contexts']),
        'sources': len(r['sources']),
        'latency_ms': r['latency_ms'],
        'error': r['error'],
    }
    for r in system_records
])
display(df_sys)

,case_id,action,contexts,sources,latency_ms,error
0,SYM-01,generate,8,7,14992.92,None
1,SYM-02,generate,7,6,12033.45,None
2,SYM-03,generate,9,4,12019.06,None
3,CAU-01,generate,7,6,12794.82,None
4,CAU-02,generate,6,5,11295.58,None
5,CAU-03,generate,7,6,9249.57,None
6,CMP-01,generate,6,5,9923.39,None
7,CMP-02,generate,7,4,10138.55,None
8,TRT-01,generate,6,4,10291.62,None
9,TRT-02,generate,6,3,9795.03,None


In [5]:
# 5. Chấm điểm bằng RAGAS (5 chỉ số cốt lõi + Safe Abstention Accuracy)
scored_records = await score_records_with_ragas(
    system_records,
    judge_model=JUDGE_MODEL,
    judge_provider=JUDGE_PROVIDER,
    embedding_model=EMBEDDING_MODEL,
    request_delay_seconds=REQUEST_DELAY_SECONDS,
)
summary = summarize_evaluation(
    scored_records,
    dataset_meta=dataset_meta,
    judge_model=f'{JUDGE_PROVIDER}/{JUDGE_MODEL}',
    embedding_model=EMBEDDING_MODEL,
    kb_build_id=kb_build_id,
)
df_summary = pd.DataFrame([
    {'metric': k, **v} for k, v in summary['ragas_summary'].items()
])
display(df_summary)
df_per_case = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'category': r['category'],
        'action': r['action'],
        **(r.get('ragas_scores') or {}),
        'abstention_ok': r.get('abstention_correct'),
        'latency_ms': r['latency_ms'],
    }
    for r in scored_records
])
display(df_per_case)
print('Safe Abstention:', summary['safe_abstention'])
print('Latency (ms):', summary['latency_ms'])
print('Token Usage:', summary['token_usage'])

Safe Abstention: {'correct': 3, 'total': 4, 'accuracy': 0.75}
Latency (ms): {'mean': 9050.8, 'p50': 9916.4, 'p95': 12794.8}
Token Usage: {'input': 72670, 'output': 3254, 'total': 75924}


,metric,mean,min,max,count
0,faithfulness,0.9330,0.7778,1.0000,9
1,answer_relevancy,0.9561,0.9450,0.9639,3
2,context_precision,0.5333,0.2667,1.0000,3
3,context_recall,0.9231,0.0000,1.0000,13
4,factual_correctness,0.3975,0.2000,0.6000,4


,case_id,category,action,faithfulness,answer_relevancy,context_precision,context_recall,factual_correctness,abstention_ok,latency_ms
0,SYM-01,symptoms_and_types,generate,1.0000,NaN,NaN,1.0,NaN,None,14992.92
1,SYM-02,symptoms_and_types,generate,0.8333,NaN,NaN,1.0,0.29,None,12033.45
2,SYM-03,symptoms_and_types,generate,NaN,NaN,NaN,1.0,NaN,None,12019.06
3,CAU-01,causes_and_mechanisms,generate,NaN,NaN,NaN,NaN,NaN,None,12794.82
4,CAU-02,causes_and_mechanisms,generate,0.7857,NaN,NaN,1.0,NaN,None,11295.58
5,CAU-03,causes_and_mechanisms,generate,1.0000,NaN,NaN,NaN,0.50,None,9249.57
6,CMP-01,comparison,generate,1.0000,NaN,1.0000,1.0,NaN,None,9923.39
7,CMP-02,comparison,generate,NaN,0.9594,0.3333,1.0,NaN,None,10138.55
8,TRT-01,treatment_and_actives,generate,NaN,0.9450,NaN,1.0,NaN,None,10291.62
9,TRT-02,treatment_and_actives,generate,NaN,NaN,NaN,0.0,0.20,None,9795.03


In [6]:
# 6. Xuất báo cáo chi tiết ra file Markdown (.md), JSON thô và hiển thị trực tiếp tại Notebook
report_path, raw_path = export_evaluation_artifacts(
    summary,
    output_dir=RESULTS_DIR,
    report_filename='ragas_evaluation_report.md',
    raw_filename='ragas_evaluation_raw.json',
)
print('Đã xuất báo cáo Markdown:', report_path)
print('Đã xuất dữ liệu thô JSON:', raw_path)
report_md = render_markdown_report(summary)
preview_md = report_md.split('## 5. Chi tiết câu trả lời')[0].strip()
display(Markdown(preview_md))

Đã xuất báo cáo Markdown: C:\Study\Acne Advisor AI\acne-agent-system\evaluation\results\ragas_evaluation_report.md
Đã xuất dữ liệu thô JSON: C:\Study\Acne Advisor AI\acne-agent-system\evaluation\results\ragas_evaluation_raw.json


# Báo cáo Đánh giá Hệ thống Acne Advisor AI (RAGAS)

- **Thời điểm chạy (UTC):** `2026-11-23T07:23:45.553228+00:00`
- **Bộ dữ liệu đánh giá:** `acne_advisor_ragas_benchmark` (20 câu hỏi: 16 câu trả lời, 4 câu từ chối an toàn)
- **Judge LLM (RAGAS):** `google/gemini-3.1-flash-lite`
- **Embedding Model:** `models/gemini-embedding-2`
- **Knowledge Build ID:** `b768fae999391f47760a`

## 1. Tổng quan chỉ số RAGAS

Phương pháp đánh giá tham chiếu bài báo *RAGAS: Automated Evaluation of Retrieval Augmented Generation* (Es et al., 2023, `arXiv:2309.15217`). Thang điểm từ `0.000` đến `1.000` (càng cao càng tốt).

| Chỉ số | Trung bình | Thấp nhất | Cao nhất | Số ca chấm | Đánh giá |
| --- | --- | --- | --- | --- | --- |
| Độ trung thực với bằng chứng (Faithfulness) | 0.933 | 0.778 | 1.000 | 9 | Tốt (>= 0.80) |
| Độ liên quan câu trả lời (Answer Relevancy) | 0.956 | 0.945 | 0.964 | 3 | Tốt (>= 0.80) |
| Độ chính xác ngữ cảnh (Context Precision) | 0.533 | 0.267 | 1.000 | 3 | Cần cải thiện (< 0.65) |
| Độ bao phủ ngữ cảnh (Context Recall) | 0.923 | 0.000 | 1.000 | 13 | Tốt (>= 0.80) |
| Độ chính xác so với đáp án chuẩn (Factual Correctness) | 0.398 | 0.200 | 0.600 | 4 | Cần cải thiện (< 0.65) |

## 2. Chỉ số Vận hành & An toàn

| Hạng mục | Kết quả |
| --- | --- |
| Tỷ lệ từ chối an toàn (Safe Abstention Accuracy) | 3/4 (0.750) |
| Độ trễ trung bình (Mean Latency) | 9050.8 ms |
| Độ trễ trung vị (P50 Latency) | 9916.4 ms |
| Độ trễ P95 (P95 Latency) | 12794.8 ms |
| Tổng token tiêu thụ (Generation) | input=72670, output=3254, total=75924 |
| Số lỗi runtime | 0 |

## 3. Phân tích theo nhóm câu hỏi

| Nhóm câu hỏi | Số ca | Faithfulness | Answer Relevancy | Context Precision | Context Recall | Factual Correctness | Độ trễ TB (ms) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `symptoms_and_types` | 3 | 0.917 | N/A | N/A | 1.000 | 0.290 | 13015.1 |
| `causes_and_mechanisms` | 3 | 0.893 | N/A | N/A | 1.000 | 0.500 | 11113.3 |
| `comparison` | 2 | 1.000 | 0.959 | 0.667 | 1.000 | N/A | 10031.0 |
| `treatment_and_actives` | 4 | 0.778 | 0.945 | 0.267 | 0.667 | 0.200 | 8264.7 |
| `safety_and_precautions` | 3 | 1.000 | 0.964 | N/A | 1.000 | 0.600 | 10759.3 |
| `lifestyle_and_care` | 1 | 1.000 | N/A | N/A | 1.000 | N/A | 9909.5 |
| `safe_abstention` | 4 | N/A | N/A | N/A | N/A | N/A | 3330.6 |

## 4. Bảng điểm chi tiết từng câu hỏi

| Case ID | Nhóm | Hành động | Faithfulness | Relevancy | Precision | Recall | Correctness | Từ chối đúng | Độ trễ (ms) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `SYM-01` | `symptoms_and_types` | `generate` | 1.000 | N/A | N/A | 1.000 | N/A | - | 14992.92 |
| `SYM-02` | `symptoms_and_types` | `generate` | 0.833 | N/A | N/A | 1.000 | 0.290 | - | 12033.45 |
| `SYM-03` | `symptoms_and_types` | `generate` | N/A | N/A | N/A | 1.000 | N/A | - | 12019.06 |
| `CAU-01` | `causes_and_mechanisms` | `generate` | N/A | N/A | N/A | N/A | N/A | - | 12794.82 |
| `CAU-02` | `causes_and_mechanisms` | `generate` | 0.786 | N/A | N/A | 1.000 | N/A | - | 11295.58 |
| `CAU-03` | `causes_and_mechanisms` | `generate` | 1.000 | N/A | N/A | N/A | 0.500 | - | 9249.57 |
| `CMP-01` | `comparison` | `generate` | 1.000 | N/A | 1.000 | 1.000 | N/A | - | 9923.39 |
| `CMP-02` | `comparison` | `generate` | N/A | 0.959 | 0.333 | 1.000 | N/A | - | 10138.55 |
| `TRT-01` | `treatment_and_actives` | `generate` | N/A | 0.945 | N/A | 1.000 | N/A | - | 10291.62 |
| `TRT-02` | `treatment_and_actives` | `generate` | N/A | N/A | N/A | 0.000 | 0.200 | - | 9795.03 |
| `TRT-03` | `treatment_and_actives` | `generate` | 0.778 | N/A | 0.267 | 1.000 | N/A | - | 9352.89 |
| `TRT-04` | `treatment_and_actives` | `abstain` | N/A | N/A | N/A | N/A | N/A | - | 3619.22 |
| `SAF-01` | `safety_and_precautions` | `generate` | 1.000 | N/A | N/A | 1.000 | N/A | - | 12528.95 |
| `SAF-02` | `safety_and_precautions` | `generate` | N/A | 0.964 | N/A | 1.000 | 0.600 | - | 10276.2 |
| `SAF-03` | `safety_and_precautions` | `generate` | 1.000 | N/A | N/A | 1.000 | N/A | - | 9472.61 |
| `CAR-01` | `lifestyle_and_care` | `generate` | 1.000 | N/A | N/A | 1.000 | N/A | - | 9909.5 |
| `ABS-01` | `safe_abstention` | `generate` | N/A | N/A | N/A | N/A | N/A | Đạt | 7.14 |
| `ABS-02` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 1737.77 |
| `ABS-03` | `safe_abstention` | `generate` | N/A | N/A | N/A | N/A | N/A | Chưa đạt | 9202.61 |
| `ABS-04` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 2375.03 |